# Variant Effect Prediction with NVIDIA Parabricks and CodonFM

#### Authors: Gary Burnett, Fan Cao 

This lab demonstrates an end-to-end computational biology pipeline for predicting the effects of genetic variants on protein-coding genes. We will leverage NVIDIA's GPU-accelerated tools and state-of-the-art deep learning models to process genomic data efficiently.

### What We'll Do

1. **Variant Calling**: We'll start with raw DNA sequencing data (FASTQ files) from an Illumina sequencer and use NVIDIA Parabricks to rapidly align these reads to the human reference genome and identify genetic variants.

2. **Gene Annotation**: We'll process the GENCODE gene annotation database to extract protein-coding sequences and map our detected variants onto specific genes and transcripts.

3. **Effect Prediction**: Finally, we'll use CodonFM, NVIDIA's foundation model for RNA sequence analysis, to predict how each variant affects the protein-coding potential of genes.

### Why This Matters

Genetic variants can have vastly different effects on human health. Some variants are harmless (synonymous), while others can significantly disrupt protein function. This pipeline combines high-performance computing with machine learning to quickly and accurately assess variant effects—a critical capability for clinical genomics and precision medicine.

### Technology Stack

- **NVIDIA Parabricks**: GPU-accelerated genomics tools for variant calling
- **CodonFM**: A foundation model trained on codon-level sequences for fine-grained variant effect prediction
- **Polars**: Efficient DataFrame processing for large genomic datasets
- **BCFTools & SAMTools**: Standard bioinformatics utilities for working with genomic data formats

Let's get started!

## 1. Variant Calling 

<p><img src="images/variant_calling.png"></p>

### The dataset

First let's download the dataset. The files are described in the next section. 

In [ ]:
%%sh 

# Create data and ref directory if it doesn't exist 
mkdir -p data/ref

# Download the FASTQ samples 
# Note: This downloads some unnecessary files and can be streamlined in the future 
wget -q -O parabricks_sample.tar.gz https://s3.amazonaws.com/parabricks.sample/parabricks_sample.tar.gz
tar xzvf parabricks_sample.tar.gz
mv parabricks_sample/Data/sample_* data 
mv parabricks_sample/Ref/* data/ref 

In this first section of the notebook, we start with FASTQ files directly off a sequencer and we want to accurately find where any genetic mutations may exist in this sample. The samples in this notebook are from an Illumina short-read sequencer, meaning that the DNA was cut into fragments of about 100 base pairs (bp) each and then run through the sequencer to determine which bases (A,T,C,G) are at which position. This information is stored in a FASTQ file, a human readable format that contains repeated groups of four lines containing a header, the sequence, a separator, and a quality score for each fragment in the sequence. We can open our FASTQ file below to see for ourselves: 

In [3]:
%%sh 

zcat data/sample_1.fq.gz | head 

@HWI-D00127:570:HK3TJBCX2:1:1101:1141:1991 1:N:0:GGCTAC
NATCACAATTCCTTTTTACCTCTCAGACCTTATTCTAACTGAAGATATTAACATTTACTTCACTTTCAAATTTTAATGCCAGGTAGAGGAAGCCAGAGCCATGCTGAAGCTCCAT
+
#00<<D?EHHIHIIIIIIIFHHHI@1FGHHIIGHHHFHIIIIHHECC<@H?@1@@GCHHHHHHIII?@GDGG?CEFE@E?FEGHHIEE?C1<1D<@FHHHHCGHHHED@@@@@GC
@HWI-D00127:570:HK3TJBCX2:1:1101:1441:1958 1:N:0:GGCTAC
NCACCCGGTACAGCTGGTCCACCTGTGCCTGGCACTGCATGTAGGCTTCATAGTCTGCAAACACCTTGAACCTGGAATGGGAGAAAGGAGGCTGGGCCAGGGCAGCAGGGAGCCC
+
#<DDDHIIIIIIIIIIIIIIIIIIIIIIIIIHIIIIIIIGIIIIIIIIIIIIIIIIIIIHIIIIIIHIIIIIHIIIIGIHIIIHHIIIIIIIHIHHHIIIHHHIIIIHIIIIIII
@HWI-D00127:570:HK3TJBCX2:1:1101:1318:1961 1:N:0:GGCTAC
NAGCGCAGAAGCATCCGTTATGGGAGAGAAGTTGAATTAAAGCCATATATTGCCGCCCACTTTGATGTCCTTCCCACTGAGTTCACCCTGGGGGATGACAAGCATTATGGTGGAT


For more information on the FASTQ format, see this paper from [Nucleic Acids Research](https://academic.oup.com/nar/article/38/6/1767/3112533?login=false) describing the initial implementation and modern updates to the format.

Sequencing DNA as fragments is a compromise, since DNA strands are so long and we do not currently have the technology to sequence the entire strand at once. Therefore, the first step in our analysis is to re-assemble these fragments back into their full sequence. This process is called alignment and requires a reference genome in order to know what order to re-assemble the fragments into. Since the reference is used as a "ground truth" sequence, we must be certain that the sequence is correct. There are several options for reference genomes to use, but a common one is the the Genome Reference Consortium Human Build 38 (GRCh38), which is a highly accurate reference genome developed by an international collaboration of researchers from NCI, Sanger, EBI, and others. 

The reference files exist in `data/ref`. There is a `.fasta` file (similar to our `.fastq` samples above) and a set of index files generated by [`bwa index`](https://bio-bwa.sourceforge.net/bwa.shtml) and [`samtools faidx`](https://www.htslib.org/doc/samtools-faidx.html) which will make searching through the reference much more efficient. These operations can take a long time to run, so they are typically pre-computed and downloaded to save time. 

In [4]:
%%sh 

ls -lh data/ref

total 8.5G
-rw-rw-r-- 1 gburnett gburnett  33M Oct 17  2024 gencode.v47.basic.annotation.gtf.gz
-rw-rw-r-- 1 gburnett gburnett  39M Feb 11 13:06 gencode.v47.basic.annotation.processed.filtered.tsv
-rw-rw-r-- 1 gburnett gburnett  19M Feb 11 06:08 gencode.v47.basic.annotation.processed.tsv
-rw-r--r-- 1 gburnett gburnett 407K Jan 28  2019 Homo_sapiens_assembly38.dict
-rw-r--r-- 1 gburnett gburnett 3.1G Jan 28  2019 Homo_sapiens_assembly38.fasta
-rw-r--r-- 1 gburnett gburnett  20K Jan 28  2019 Homo_sapiens_assembly38.fasta.amb
-rw-r--r-- 1 gburnett gburnett 445K Jan 28  2019 Homo_sapiens_assembly38.fasta.ann
-rw-r--r-- 1 gburnett gburnett 3.0G Jan 28  2019 Homo_sapiens_assembly38.fasta.bwt
-rw-r--r-- 1 gburnett gburnett 158K Jan 28  2019 Homo_sapiens_assembly38.fasta.fai
-rw-r--r-- 1 gburnett gburnett 768M Jan 28  2019 Homo_sapiens_assembly38.fasta.pac
-rw-r--r-- 1 gburnett gburnett 1.5G Jan 28  2019 Homo_sapiens_assembly38.fasta.sa
-rw-r--r-- 1 gburnett gburnett  59M Jan 28  2019 Homo_sap

### Alignment and variant calling methods

There are several algorithms that can perform alignment, but we will stick to the commonly used [BWA aligner](https://github.com/lh3/bwa) based on the [Burrows-Wheeler Transform](https://sandbox.bio/concepts/bwt). For specific implementation details see the [original paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC2705234/). 

Once the sample is aligned, we can use a variant caller to determine which locations in the sample differ from the reference genome. In this lab we use DeepVariant as our variant caller because it is highly accurate. DeepVariant was [originaly developed by Google](https://research.google/blog/deepvariant-highly-accurate-genomes-with-deep-neural-networks/) in 2017 as a method to use deep learning for variant calling. It works by converting the reads into images, and then using a convolutional neural network (CNN) to predict variants. This method won multiple awards for best accuracy during the 2020 Precision FDA Truth challenge. 

### GPU-accelerated implementation

We will use GPU-accelerated versions of BWA and DeepVariant through [NVIDIA Parabricks](https://docs.nvidia.com/clara/parabricks/latest/index.html). Parabricks is a suite of GPU-accelerated tools for genomics analysis. It is designed as a drop-in replacement for many of the CPU versions of tools that are commonly used today. The accuracy is guaranteed to be the same as these tools, but because they run on the GPU they can be orders of magnitude faster. This is especially helpful when you have large amounts of data that need to be processed. 

The Parabricks `deepvariant_germline` pipeline as shown in the diagram below takes in FASTQ files and a reference as input, runs GPU-accelerated BWA and then GPU-accelerated DeepVariant all in one step. This combined command allows for compute to be shared between multiple steps, ensuring faster processing that running the tools as separate commands. 

<p><img src="images/dv_diagram.webp"></p>

*Source: Parabricks `deepvariant_germline` [documentation](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_deepvariant_germline.html)*

Use the next cell to run this pipeline and generate the variants we will use downstream. 

In [ ]:
%%sh

# Make the output directory if it doesn't already exist 
mkdir -p data/out

# Run Parabricks DeepVariant on our FASTQ samples 
docker run \
    --rm \
    --gpus all \
    --volume $(pwd):/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun deepvariant_germline \
    --ref data/ref/Homo_sapiens_assembly38.fasta \
    --in-fq data/sample_1.fq.gz data/sample_2.fq.gz \
    --out-variants data/out/deepvariant.vcf.gz \
    --out-bam data/out/sample.bam 

The output of DeepVariant is a Variant Call Format (VCF) file which represents all the variants found in the sample in a tabular format. The column names are standardized as follows: 

| Column Name | Value |
|---|---|
| CHROM | Reference sequence name (chromosome or contig) |
| POS | 1-based leftmost position of the variant on CHROM |
| ID | Variant identifier(s) (e.g., rsID) or `.` if none |
| REF | Reference allele sequence at POS |
| ALT | Comma-separated alternate allele(s) observed at POS |
| QUAL | Phred-scaled quality score for the assertion of the ALT allele(s) |
| FILTER | `PASS` if variant passed filters, or semicolon-separated failing filter names |
| INFO | Semicolon-separated annotations (flags or key=value pairs) with additional site-level metadata |
| FORMAT | Colon-separated keys defining the per-sample fields (e.g., `GT:DP:AD`) |
| sample | Sample-specific values corresponding to FORMAT (e.g., genotype, depth, allele depths)

This file also contains a header with meta data about the samples. For more information on the VCF file format, review the [spec](https://samtools.github.io/hts-specs/VCFv4.2.pdf). 

[bcftools](https://github.com/samtools/bcftools) is a powerful command line utility to view and manipulate vcf files. Use [bcftools view](https://samtools.github.io/bcftools/bcftools.html#view) to inspect the contents of our VCF file. 

In [9]:
%%sh

# Look at the first few lines of the VCF file (without the header) 
bcftools view --no-header data/out/deepvariant.vcf.gz | head

chr1	10261	.	T	C	0.3	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:11:3:1,2:0.666667:0,16,11
chr1	10273	.	T	C	0.3	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:11:3:1,2:0.666667:0,16,12
chr1	10279	.	T	C	0.2	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:13:3:1,2:0.666667:0,17,14
chr1	16378	.	T	C	8.1	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:6:2:0,2:1:6,8,0
chr1	63268	.	T	C	15.8	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:15:5:0,5:1:15,21,0
chr1	63516	.	A	G	13.4	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:13:47:0,47:1:13,20,0
chr1	63527	.	T	C	23.3	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:20:42:0,42:1:23,23,0
chr1	131310	.	G	C	0.7	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:8:12:8,4:0.333333:0,8,13
chr1	131315	.	T	C	4	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:2:14:10,4:0.285714:0,1,0
chr1	131552	.	G	T	2.8	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:3:17:9,8:0.470588:0,1,7


### Filter VCF

For our task, the VCF includes a lot of unnecessary information. Let's subset the VCF to only include columns we will need later, and to only include rows that pass all the filter criteria. 

To achieve this we use [bcftools query](https://samtools.github.io/bcftools/bcftools.html#query) with the following arguments: 

| Argument | Value |
|---|---|
| `-H` | Print the header. |
| `-f` | Only include certain columns. |
| `-i` | Only include sites that satisfy this criteria. |

Then we zip the output and store it as a new tab-separated file.  

In [14]:
%%sh 

bcftools query -H -f '%CHROM\t%POS\t%REF\t%ALT\n' \
  -i 'TYPE="snp" && FILTER="PASS"' \
  "data/out/deepvariant.vcf.gz" | 
  bgzip > data/out/deepvariant.filtered.tsv.gz

This new file should have fewer columns: 

In [15]:
%%sh 

zcat data/out/deepvariant.filtered.tsv.gz | head

#[1]CHROM	[2]POS	[3]REF	[4]ALT
chr1	16378	T	C
chr1	63268	T	C
chr1	63516	A	G
chr1	63527	T	C
chr1	133483	G	T
chr1	268516	C	T
chr1	268559	A	C
chr1	414574	A	G
chr1	630026	C	T


## 2. Gene Annotation

<p><img src="images/gene_annotation.png"></p>

The [GENCODE project](https://www.gencodegenes.org/pages/gencode.html) was launched by the National Human Genome Research Institute (NHGRI) and aims to annotate all the functional elements in the human genome. It is widely used for bioinformatics and accepted as a gold standard annotation. The annotations are stored in Gene Transfer Format (GTF) format. Each line in a GTF file has the following 9 fields: 

| Field | Description |
|---|---|
| seqname | the chromosome name |
| source | the program that generated the features |
| feature | feature type (e.g., gene, transcript, exon, CDS, UTR, variation) |
| start | 1-based start position of the feature |
| end | 1-based end position of the feature |
| score | floating point score (use `.` if not applicable) |
| strand | `+` for forward strand, `-` for reverse strand |
| frame | reading frame: 0, 1, or 2 indicating which base is the start of the codon (use `.` if not applicable) |
| attribute | semicolon-separated key=value (or key "value") pairs with additional info (e.g., `gene_id`, `transcript_id`, `gene_name`) 

In this notebook, we use the GENCODE Human Release 47 for GRCh38 basic annotation set. Read more about this file on the [GENCODE website](https://www.gencodegenes.org/human/release_47.html). 

In [ ]:
%%sh 

GENCODE_FILE_URL="https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.basic.annotation.gtf.gz"

# Download the gtf file 
wget -q -P data/ref ${GENCODE_FILE_URL}

Inspect the contents of the GTF file. 

In [5]:
%%sh 

# Inspect the GTF file 
zcat data/ref/gencode.v47.basic.annotation.gtf.gz | head 

##description: evidence-based annotation of the human genome (GRCh38), version 47 (Ensembl 113)
##provider: GENCODE
##contact: gencode-help@ebi.ac.uk
##format: gtf
##date: 2024-07-19
chr1	HAVANA	gene	11121	24894	.	+	.	gene_id "ENSG00000290825.2"; gene_type "lncRNA"; gene_name "DDX11L16"; level 2; tag "overlaps_pseudogene";
chr1	HAVANA	transcript	11426	14409	.	+	.	gene_id "ENSG00000290825.2"; transcript_id "ENST00000832828.1"; gene_type "lncRNA"; gene_name "DDX11L16"; transcript_type "lncRNA"; transcript_name "DDX11L16-264"; level 2; tag "basic"; tag "TAGENE";
chr1	HAVANA	exon	11426	11671	.	+	.	gene_id "ENSG00000290825.2"; transcript_id "ENST00000832828.1"; gene_type "lncRNA"; gene_name "DDX11L16"; transcript_type "lncRNA"; transcript_name "DDX11L16-264"; exon_number 1; exon_id "ENSE00004248702.1"; level 2; tag "basic"; tag "TAGENE";
chr1	HAVANA	exon	12010	12227	.	+	.	gene_id "ENSG00000290825.2"; transcript_id "ENST00000832828.1"; gene_type "lncRNA"; gene_name "DDX11L16"; transcript_typ

In `utils.py` there is a script called `process_gtf_file` which filters for protein-coding genes only and outputs a tab-separated `.tsv` file. The important parts of this file for our purposes include the gene names with their locations in the reference genome. 

In [20]:
from utils.gtf_processing import process_gtf_file 

REFERENCE_DIR = "data/ref"
gtf_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.gtf.gz"
out_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.processed.tsv"

_ = process_gtf_file(gtf_file, out_file)

Open the processed `GTF` file 

In [6]:
%%sh 

head data/ref/gencode.v47.basic.annotation.processed.tsv

gene_id	name	chrom	strand	txStart	txEnd	cdsStart	cdsEnd	exon_count	exonStarts	exonEnds	gene_name	transcript_type	is_canonical	is_mane_select
ENSG00000186092.7	ENST00000641515.2	chr1	+	65418	71585	65564	70008	3	65564,69036,	65573,70008,	OR4F5	protein_coding	true	true
ENSG00000284733.2	ENST00000426406.4	chr1	-	450739	451678	450739	451678	1	450739,	451678,	OR4F29	protein_coding	true	true
ENSG00000284662.2	ENST00000332831.5	chr1	-	685715	686654	685715	686654	1	685715,	686654,	OR4F16	protein_coding	true	true
ENSG00000187634.13	ENST00000618323.5	chr1	+	923922	944574	924431	944153	9	924431,925921,930154,931038,935771,939039,939271,941143,942135,942409,942558,943252,943697,943907,	924948,926013,930336,931089,935896,939129,939412,941306,942251,942488,943058,943377,943808,944153,	SAMD11	protein_coding	false	false
ENSG00000187634.13	ENST00000616016.5	chr1	+	923922	944574	924431	944153	9	924431,925921,930154,931038,935771,939039,939274,941143,942135,942409,942558,943252,943697,943907,	924948,92601

### Extract coding sequence (CDS)

`extract_cds` is a function that uses the CDS start and stop positions from the GTF file and maps them into the reference file to extract the full CDS sequence. It will also gather information about start and stop codons. 

In [21]:
from utils.extract_cds import extract_cds

reference_genome = f"{REFERENCE_DIR}/Homo_sapiens_assembly38.fasta"
annotation_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.processed.tsv"
filtered_transcripts_file = f"{REFERENCE_DIR}/gencode.v47.basic.annotation.processed.filtered.tsv"

extract_cds(reference_genome, annotation_file, filtered_transcripts_file)

Reference genome: data/ref/Homo_sapiens_assembly38.fasta
Annotation file: data/ref/gencode.v47.basic.annotation.processed.tsv
Output file: data/ref/gencode.v47.basic.annotation.processed.filtered.tsv
Loaded 23 chromosomes from data/ref/Homo_sapiens_assembly38.fasta
Loaded 64,488 transcripts from data/ref/gencode.v47.basic.annotation.processed.tsv
After deduplicating by genomic structure: 51,650 transcripts
Extracting CDS sequences...
Running quality checks...

CDS Quality Summary (before filtering):
Total transcripts: 51,650
Has start codon (ATG): 51,419 (99.6%)
Has stop codon (TAA/TAG/TGA): 51,341 (99.4%)
Length divisible by 3: 51,573 (99.9%)
Has internal stop codons: 113 (0.2%)
All quality criteria met: 51,061

After quality filtering: 51,061 transcripts
After filtering to canonical transcripts: 19,407 transcripts
After canonical filter + CDS deduplication: 19,310 unique transcripts
  (Removed 31,751 transcripts)
Saved 19,310 unique transcripts to data/ref/gencode.v47.basic.annotatio

Open the filtered `.tsv` file and notice the additional columns for information about the CDS sequence and codons. 

In [ ]:
%%sh 

head data/ref/gencode.v47.basic.annotation.processed.filtered.tsv

gene_id	name	chrom	strand	txStart	txEnd	cdsStart	cdsEnd	exon_count	exonStarts	exonEnds	gene_name	transcript_type	is_canonical	is_mane_select	cds_sequence	has_start_codon	has_stop_codon	length_divisible_by_3	has_internal_stop_codons	cds_length
ENSG00000186092.7	ENST00000641515.2	chr1	+	65418	71585	65564	70008	3	65564,69036,	65573,70008,	OR4F5	protein_coding	true	true	ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGAGTGAAACGAATAACTCTATGGTGACTGAATTCATTTTTCTGGGTCTCTCTGATTCTCAGGAACTCCAGACCTTCCTATTTATGTTGTTTTTTGTATTCTATGGAGGAATCGTGTTTGGAAACCTTCTTATTGTCATAACAGTGGTATCTGACTCCCACCTTCACTCTCCCATGTACTTCCTGCTAGCCAACCTCTCACTCATTGATCTGTCTCTGTCTTCAGTCACAGCCCCCAAGATGATTACTGACTTTTTCAGCCAGCGCAAAGTCATCTCTTTCAAGGGCTGCCTTGTTCAGATATTTCTCCTTCACTTCTTTGGTGGGAGTGAGATGGTGATCCTCATAGCCATGGGCTTTGACAGATATATAGCAATATGCAAGCCCCTACACTACACTACAATTATGTGTGGCAACGCATGTGTCGGCATTATGGCTGTCACATGGGGAATTGGCTTTCTCCATTCGGTGAGCCAGTTGGCGTTTGCCGTGCACTTACTCTTCTGTGGTCCCAATGAGGTCGATAGTTTTTATTGTGACCTTCCTAGGGTAATCAAACTTGCCTGTACAGATACCTACAGGCTAGATA

### Put it all together 

Lastly, use Python to load this data into a [Polars dataframe](https://docs.pola.rs/py-polars/html/reference/dataframe/index.html). Polars is like Pandas except much more memory efficient for large datasets due to using the Apache Arrow memory format as opposed to Numpy arrays. For more information on the differences, check out the [Polars documentation](https://docs.pola.rs/user-guide/migration/pandas/). 

In [8]:
import polars as pl 

We have one table that shows the sequence for protein coding region and its index in the genome. 

In [3]:
gencode_v47_file = "data/ref/gencode.v47.basic.annotation.processed.filtered.tsv"
genes = pl.read_csv(gencode_v47_file, separator="\t")
genes = genes.with_row_index('id')
genes = genes.with_columns(pl.col("id").cast(pl.Int64))
genes.head()

id,gene_id,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exon_count,exonStarts,exonEnds,gene_name,transcript_type,is_canonical,is_mane_select,cds_sequence,has_start_codon,has_stop_codon,length_divisible_by_3,has_internal_stop_codons,cds_length
i64,str,str,str,str,i64,i64,i64,i64,i64,str,str,str,str,bool,bool,str,bool,bool,bool,bool,i64
0,"""ENSG00000186092.7""","""ENST00000641515.2""","""chr1""","""+""",65418,71585,65564,70008,3,"""65564,69036,""","""65573,70008,""","""OR4F5""","""protein_coding""",true,true,"""ATGAAGAAGGTAACTGCAGAGGCTATTTCC…",true,true,true,false,981
1,"""ENSG00000284733.2""","""ENST00000426406.4""","""chr1""","""-""",450739,451678,450739,451678,1,"""450739,""","""451678,""","""OR4F29""","""protein_coding""",true,true,"""ATGGATGGAGAGAATCACTCAGTGGTATCT…",true,true,true,false,939
2,"""ENSG00000187634.13""","""ENST00000616016.5""","""chr1""","""+""",923922,944574,924431,944153,9,"""924431,925921,930154,931038,93…","""924948,926013,930336,931089,93…","""SAMD11""","""protein_coding""",true,true,"""ATGCCGGCGGTCAAGAAGGAGTTCCCGGGC…",true,true,true,false,2535
3,"""ENSG00000188976.11""","""ENST00000327044.7""","""chr1""","""-""",944202,959256,944693,959240,9,"""944693,945056,945517,946172,94…","""944800,945146,945653,946286,94…","""NOC2L""","""protein_coding""",true,true,"""ATGGCAGCTGCGGGGAGCCGCAAGAGGCGC…",true,true,true,false,2250
4,"""ENSG00000187961.15""","""ENST00000338591.8""","""chr1""","""+""",960583,965719,960693,965191,9,"""960693,961292,961628,961825,96…","""960800,961552,961750,962047,96…","""KLHL17""","""protein_coding""",true,true,"""ATGCAGCCCCGCAGCGAGCGCCCGGCCGGC…",true,true,true,false,1929


And we have another table with all the variants found in the samples and their index in the genome. 

In [4]:
variants = pl.read_csv("data/out/deepvariant.filtered.tsv.gz", separator='\t', has_header=True)

# Rename columns to simplify (Ex. #[1]CHROM > chrom)
variants.columns = ['chrom', 'pos', 'ref', 'alt'] 

# Remove any positions with multiple alternate alleles 
variants = variants.filter(pl.col('alt').str.contains(r'^[ATCG]$')) 

variants.head()

chrom,pos,ref,alt
str,i64,str,str
"""chr1""",16378,"""T""","""C"""
"""chr1""",63268,"""T""","""C"""
"""chr1""",63516,"""A""","""G"""
"""chr1""",63527,"""T""","""C"""
"""chr1""",133483,"""G""","""T"""


We can join these tables to find which transcripts have variants and where in the CDS sequence they are. This join requires helper functions. First we use `map_variants_to_genes_by_exons_efficient` to create a map of each transcript and which variants it contains. 

In [5]:
from utils.extract_cds import map_variants_to_genes_by_exons_efficient 

# Calculate which genes have variants 
gene_variant_mapping = map_variants_to_genes_by_exons_efficient(genes, variants, variant_columns=['pos', 'ref', 'alt']) 

Processing 19310 genes and 113415 variants...


In [6]:
gene_variant_mapping

{0: {'variants': []},
 1: {'variants': []},
 2: {'variants': []},
 3: {'variants': []},
 4: {'variants': []},
 5: {'variants': [{'pos': 973858,
    'ref': 'G',
    'alt': 'C',
    'chrom': 'chr1',
    'exon_start': 973832,
    'exon_end': 974051,
    'dist_left': 1459,
    'dist_in_cds': 1459,
    'ref_codon': 'CGT',
    'alt_codon': 'CCT'}]},
 6: {'variants': []},
 7: {'variants': []},
 8: {'variants': []},
 9: {'variants': [{'pos': 1046551,
    'ref': 'A',
    'alt': 'G',
    'chrom': 'chr1',
    'exon_start': 1046396,
    'exon_end': 1046735,
    'dist_left': 3065,
    'dist_in_cds': 3065,
    'ref_codon': 'TCA',
    'alt_codon': 'TCG'}]},
 10: {'variants': []},
 11: {'variants': []},
 12: {'variants': []},
 13: {'variants': []},
 14: {'variants': []},
 15: {'variants': []},
 16: {'variants': []},
 17: {'variants': []},
 18: {'variants': []},
 19: {'variants': []},
 20: {'variants': []},
 21: {'variants': []},
 22: {'variants': []},
 23: {'variants': []},
 24: {'variants': []},
 25:

Then we use `convert_gene_variant_mapping_to_df` to put map into a table for easier post-processing. 

In [27]:
from utils.extract_cds import convert_gene_variant_mapping_to_df

# Convert map to data frame 
gene_variants = convert_gene_variant_mapping_to_df(gene_variant_mapping, genes, ['pos', 'ref', 'alt'])

# Save the output to a CSV 
gene_variants.write_csv("data/out/gene_variants.csv")

gene_variants.head()

id,chrom,pos,ref,alt,ref_codon,alt_codon,gene_name,gene_id,ref_seq,strand,codon_position,var_rel_dist_in_cds,alt_seq
u32,str,i64,str,str,str,str,str,str,str,str,i64,i64,str
0,"""chr1""",973858,"""G""","""C""","""CGT""","""CCT""","""PLEKHN1""","""ENSG00000187583.11""","""ATGGGGAACAGCCACTGTGTCCCTCAGGCC…","""+""",486,1459,"""ATGGGGAACAGCCACTGTGTCCCTCAGGCC…"
1,"""chr1""",1046551,"""A""","""G""","""TCA""","""TCG""","""AGRN""","""ENSG00000188157.16""","""ATGGCCGGCCGGTCCCACCCGGGCCCGCTG…","""+""",1021,3065,"""ATGGCCGGCCGGTCCCACCCGGGCCCGCTG…"
2,"""chr1""",1342153,"""T""","""C""","""CCA""","""CCG""","""DVL1""","""ENSG00000107404.21""","""ATGGCGGAGACCAAGATTATCTACCACATG…","""-""",121,365,"""ATGGCGGAGACCAAGATTATCTACCACATG…"
3,"""chr1""",1390230,"""C""","""A""","""GTG""","""TTG""","""CCNL2""","""ENSG00000221978.13""","""ATGGCGGCGGCGGCGGCGGCGGCTGGTGCT…","""-""",335,1005,"""ATGGCGGCGGCGGCGGCGGCGGCTGGTGCT…"
4,"""chr1""",1645163,"""T""","""C""","""GCA""","""GCG""","""CDK11B""","""ENSG00000248333.9""","""ATGGGTGATGAAAAGGACTCTTGGAAAGTG…","""-""",197,593,"""ATGGGTGATGAAAAGGACTCTTGGAAAGTG…"


We will use this an input to codonFM. 

## 3. Variant Effect Prediction

<p><img src="images/variant_effect_prediction.png"></p>

CodonFM is an open-source foundation model developed by NVIDIA for RNA sequence analysis and design. It is part of the Clara open model family, which provides tools and models for biology, chemistry, and human health applications. CodonFM is trained on the codons, sets of three nucleotides, which is more fine grained that most amino-acid level protein models. This means it can differentiate between synonymous codons, leading to more accurate results. Check out the CodonFM [blog](https://developer.nvidia.com/blog/introducing-the-codonfm-open-model-for-rna-design-and-analysis), the NVIDIA Digital Biology Examples repo on [GitHub](https://github.com/NVIDIA-Digital-Bio/CodonFM/tree/main), and the codonFM [preprint](https://research.nvidia.com/labs/dbr/assets/data/manuscripts/nv-codonfm-preprint.pdf) for more information. 

### Import libraries

In [2]:
from utils.codonfm_helpers import load_dataset
from utils.codonfm_helpers import run_mutation_predictions
from utils.codonfm_helpers import load_encodon_inference_model

import os
import torch
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name()}")

/home/gburnett/x-hx-19-v1/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully!
PyTorch version: 2.5.1+cu124
CUDA available: True
GPU device: NVIDIA A100 80GB PCIe


### Download the model 

CodonFM comes in three sizes: 80M, 600M, and 1B. For this lab, we choose the small 80M parameter NV-CodonFM-Encodon model from [Hugging Face](https://huggingface.co/nvidia/NV-CodonFM-Encodon-TE-80M-v1) so we don't waste too much lab time. 

In [ ]:
%%sh 

wget -q https://huggingface.co/nvidia/NV-CodonFM-Encodon-80M-v1/resolve/main/NV-CodonFM-Encodon-80M-v1.safetensors
wget -q https://huggingface.co/nvidia/NV-CodonFM-Encodon-80M-v1/resolve/main/config.json

### Load the model onto the GPU

In [3]:
ckpt_path = "./"
checkpoint_paths = {
    "80M": f"{ckpt_path}/NV-CodonFM-Encodon-80M-v1.safetensors",
    "600m": f"{ckpt_path}/NV-CodonFM-Encodon-600M-v1.safetensors",
    "1B": f"{ckpt_path}/NV-CodonFM-Encodon-1B-v1.safetensors",
    "1B_cdwt": f"{ckpt_path}/NV-CodonFM-Encodon-Cdwt-1B-v1.safetensors"
}

model_loaded = False
encodon_models = {}

for size, checkpoint_path in checkpoint_paths.items():
    if os.path.exists(checkpoint_path):
        try:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            model = load_encodon_inference_model(checkpoint_path, device=device)
            
            # Extract model name from path
            model_name = os.path.basename(os.path.dirname(os.path.dirname(checkpoint_path)))
            display_name = f"EnCodon ({size})"
            
            encodon_models[display_name] = {
                'model': model,
                'path': checkpoint_path,
                'device': device
            }
            print(f"✅ Successfully loaded {display_name} from: {checkpoint_path}")
            model_loaded = True
        except Exception as e:
            print(f"Failed to load from {checkpoint_path}: {e}")
            continue

if not model_loaded:
    print("❌ Could not load any Encodon model from the specified paths.")
    print("Please ensure a checkpoint exists or update the checkpoint_paths list.")
else:
    print(f"\n✅ Loaded {len(encodon_models)} models: {list(encodon_models.keys())}")

Loading Encodon model from: .//NV-CodonFM-Encodon-80M-v1.safetensors
✅ Model loaded successfully on cuda
Model parameters: 76,833,861
Tokenizer vocabulary size: 69
✅ Successfully loaded EnCodon (80M) from: .//NV-CodonFM-Encodon-80M-v1.safetensors

✅ Loaded 1 models: ['EnCodon (80M)']


### Load the dataset

This is the `results.csv` output from the previous section. 

In [4]:
# Load ClinVar Alphamissense dataset
DATASET_CONFIG = {
    'name': 'FASTQ Dataset',
    'data_path': 'data/out/gene_variants.csv',
}

# Load the dataset
dataset = load_dataset(DATASET_CONFIG)
print(f"\n📊 Dataset loaded: {dataset is not None}")

Loading FASTQ Dataset dataset...
Path: data/out/gene_variants.csv
✅ Loaded 2647 variants
Shape: (2647, 14)
Columns: ['id', 'chrom', 'pos', 'ref', 'alt', 'ref_codon', 'alt_codon', 'gene_name', 'gene_id', 'ref_seq', 'strand', 'codon_position', 'var_rel_dist_in_cds', 'alt_seq']
✅ All required columns present

Sample data:
   id ref_codon alt_codon  codon_position
0   0       CGT       CCT             486
1   1       TCA       TCG            1021
2   2       CCA       CCG             121

📊 Dataset loaded: True


### Run mutation prediction 

With the model and the dataset loaded, we are now able to run the codonFM model with our data. 

In [5]:
# Run predictions if models and dataset are available
if 'encodon_models' in locals() and 'dataset' in locals() and dataset is not None:
    predictions = run_mutation_predictions(encodon_models, dataset)
    print(f"\n✅ Predictions completed for {len(predictions)} models")
else:
    print("❌ Cannot run predictions - missing models or dataset")
    predictions = {}


=== RUNNING MUTATION PREDICTIONS FOR CLINVAR ALPHAMISSENSE ===

--- Processing EnCodon (80M) ---


EnCodon (80M) predictions: 100%|██████████| 83/83 [01:09<00:00,  1.20it/s]

✅ Completed 2647 predictions
Likelihood ratio range: [-16.504, 24.886]
🔄 Offloaded EnCodon (80M) from GPU
🧹 Cleared GPU cache after EnCodon (80M)

✅ Predictions completed for 1 models


This will generate a dictionary of log likelihood ratios (LLR) between the reference codon and the alt codon for each mutation. A positive LLR means the mutation is potentiall disruptive, and a negative LLR suggets it is tolerated.  

In [6]:
predictions

{'EnCodon (80M)': {'ids': array([   0,    1,    2, ..., 2644, 2645, 2646], shape=(2647,)),
  'likelihood_ratios': array([-2.672325  ,  0.40199184, -0.42734098, ...,  5.336752  ,
         -1.1456623 , -1.1701899 ], shape=(2647,), dtype=float32)}}

Finally we can join the predictions with our `gene_variants` table and sort the log likelihood ratio to see which variants have the largest effect on their respective genes. 

In [ ]:
pred_df = pl.DataFrame(predictions['EnCodon (80M)'], schema=['id', 'likelihood_ratio'])
pred_df = pred_df.with_columns(pl.col("id").cast(pl.UInt32))
pred_df = pred_df.join(gene_variants, on="id")
pred_df = pred_df.sort("likelihood_ratio", descending=True)
pred_df.head()

Now we have a `likelihood_ratio` column with the rest of the data in the `gene_variants` table. 

## Conclusion

In this lab we performed an end-to-end variant effect prediction pipeline. Starting from FASTQ samples we ran Parabricks BWA and DeepVariant to align to the reference and detect variants. We then used Python to manipulate a GTF file to show us the protein-coding sequence for each gene. Lastly we fed this information into CodonFM to predict the effect of each variant on the annotated genes. The dataset we used today is small but the tools are capable of handling much larger datasets thanks to GPU acceleration. 

Try running this notebook on your own FASTQ files, with the larger Encodon models, or visit the [NVIDIA BioNeMo Framework recipes](https://github.com/NVIDIA/bionemo-framework/tree/main/bionemo-recipes/recipes/codonfm_ptl_te) to retrain the CodonFM model for even more personalized results. 